In [1]:
import argparse

import torch
import numpy as np

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color, draw_geometries
from geotransformer.utils.registration import compute_registration_error
from geotransformer.modules.ops import point_to_node_partition, index_select
from geotransformer.modules.registration import get_node_correspondences

from config import make_cfg
from model import create_model

import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Specify the desired GPU index (e.g., system GPU 1)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [3]:
REF_NUM = 10

In [4]:
#FACES_FOLDER =  'faces'
FACES_FOLDER = 'faces_downsampled'

In [5]:
SRC_FILE = f"../../data/{FACES_FOLDER}/demo/src_{REF_NUM}.npy"
REF_FILE = f"../../data/{FACES_FOLDER}/demo/ref_{REF_NUM}.npy"
GT_FILE = f"../../data/{FACES_FOLDER}/demo/gt_{REF_NUM}.npy"
#WEIGHTS = "../../output/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn.17/snapshots/epoch-5.pth.tar"
WEIGHTS = "../../output/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn.old/snapshots/epoch-26.pth.tar"
#WEIGHTS = "../../output/geotransformer.facesdownsampled10.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-2.pth.tar"
#WEIGHTS = "../../output/geotransformer.facesdownsampled10mathch.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-1.pth.tar"

In [6]:
def load_data():
    src_points = np.load(SRC_FILE)
    ref_points = np.load(REF_FILE)
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
    }

    if GT_FILE is not None:
        transform = np.load(GT_FILE)
        data_dict["transform"] = transform.astype(np.float32)

    return data_dict

def open3d_webrtc_draw(geometries):
    o3d.visualization.draw(geometries)
   

    
   

In [7]:
cfg = make_cfg()
model = create_model(cfg).cuda()
state_dict = torch.load(WEIGHTS)
model.load_state_dict(state_dict["model"])

AcceleratorError: CUDA error: invalid device ordinal
GPU device may be out of range, do you have enough GPUs?
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
data_dict = load_data()

In [ ]:
data_dict.keys()

In [ ]:
data_dict['ref_points'].shape

In [ ]:
data_dict['src_points'].shape

In [ ]:
print(data_dict['ref_feats'])
print(data_dict['ref_feats'].shape)

In [ ]:
print(data_dict['src_feats'])
print(data_dict['src_feats'].shape)

In [ ]:
# Plot src_points vs ref_points
ref_pcd = make_open3d_point_cloud(data_dict['ref_points'])
src_pcd = make_open3d_point_cloud(data_dict['src_points'])

ref_pcd.paint_uniform_color(get_color('red'))    # reference in red
src_pcd.paint_uniform_color(get_color('green'))  # source in green

o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [ ]:
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)
data_dict = to_cuda(data_dict)


In [ ]:
data_dict.keys()

In [ ]:
backbone = model.backbone
transformer = model.transformer
coarse_target = model.coarse_target

In [ ]:
print(f"transform: {data_dict['transform'].shape}")
print(f"features: {data_dict['features'].shape}")
print(f"points: {len(data_dict['points'])}")
for p_id, point_data in enumerate(data_dict['points']):
    print(f"points {p_id}: {point_data.shape}")
print(f"lengths: {data_dict['lengths']}")

In [ ]:
 # Downsample point clouds
output_dict = {}    
feats = data_dict['features'].detach()
transform = data_dict['transform'].detach()

ref_length_c = data_dict['lengths'][-1][0].item()
ref_length_f = data_dict['lengths'][1][0].item()
ref_length = data_dict['lengths'][0][0].item()
points_c = data_dict['points'][-1].detach()
points_f = data_dict['points'][1].detach()
points = data_dict['points'][0].detach()

ref_points_c = points_c[:ref_length_c]
src_points_c = points_c[ref_length_c:]
ref_points_f = points_f[:ref_length_f]
src_points_f = points_f[ref_length_f:]
ref_points = points[:ref_length]
src_points = points[ref_length:]

output_dict['ref_points_c'] = ref_points_c
output_dict['src_points_c'] = src_points_c
output_dict['ref_points_f'] = ref_points_f
output_dict['src_points_f'] = src_points_f
output_dict['ref_points'] = ref_points
output_dict['src_points'] = src_points

# 1. Generate ground truth node correspondences
_, ref_node_masks, ref_node_knn_indices, ref_node_knn_masks = point_to_node_partition(
    ref_points_f, ref_points_c, model.num_points_in_patch
)
_, src_node_masks, src_node_knn_indices, src_node_knn_masks = point_to_node_partition(
    src_points_f, src_points_c, model.num_points_in_patch
)


In [ ]:
ref_points_c.shape

In [ ]:
src_points_c.shape

In [ ]:
# Initialize node KNN points
ref_padded_points_f = torch.cat([ref_points_f, torch.zeros_like(ref_points_f[:1])], dim=0)
src_padded_points_f = torch.cat([src_points_f, torch.zeros_like(src_points_f[:1])], dim=0)
ref_node_knn_points = index_select(ref_padded_points_f, ref_node_knn_indices, dim=0)
src_node_knn_points = index_select(src_padded_points_f, src_node_knn_indices, dim=0)

print(f"Ref node KNN points shape: {ref_node_knn_points.shape}")

### Initialize Node KNN Points

We need to gather the fine-level points corresponding to each superpoint (node). This is done using the KNN indices computed by `point_to_node_partition`.

## 2. Feature Extraction (KPConv Backbone)

The model uses a Kernel Point Convolution (KPConv) Feature Pyramid Network (FPN) as the backbone. This extracts features at multiple resolutions.
- `feats_c`: Coarse features used for superpoint matching.
- `feats_f`: Fine features used for refinement.

Reference: Section 3.1 of the paper.

In [ ]:
# 2. KPFCNN Encoder
feats_list = model.backbone(feats, data_dict)

feats_c = feats_list[-1]
feats_f = feats_list[0]

print(f"Coarse features shape: {feats_c.shape}")
print(f"Fine features shape: {feats_f.shape}")

## 3. Geometric Transformer

The Geometric Transformer module applies self-attention (within each point cloud) and cross-attention (between point clouds) to the coarse features. This encodes global context and geometric structure, making the features distinct and robust for matching.

- `ref_feats_c`, `src_feats_c`: Input coarse features.
- `ref_feats_c_norm`, `src_feats_c_norm`: Output transformed and normalized features.

Reference: Section 3.2 of the paper.

In [ ]:
# 3. Conditional Transformer
ref_feats_c = feats_c[:ref_length_c]
src_feats_c = feats_c[ref_length_c:]

print(f"Ref coarse features input: {ref_feats_c.shape}")
print(f"Src coarse features input: {src_feats_c.shape}")

ref_feats_c, src_feats_c = model.transformer(
    ref_points_c.unsqueeze(0),
    src_points_c.unsqueeze(0),
    ref_feats_c.unsqueeze(0),
    src_feats_c.unsqueeze(0),
)
ref_feats_c_norm = torch.nn.functional.normalize(ref_feats_c.squeeze(0), p=2, dim=1)
src_feats_c_norm = torch.nn.functional.normalize(src_feats_c.squeeze(0), p=2, dim=1)

print("Transformer output normalized.")
print(f"Ref coarse features output: {ref_feats_c_norm.shape}")
print(f"Src coarse features output: {src_feats_c_norm.shape}")

## 4. Superpoint Matching (Coarse Level)

We match the superpoints (coarse points) based on the similarity of their transformed features.
- `ref_node_corr_indices`, `src_node_corr_indices`: Indices of matched superpoints.

Reference: Section 3.3 of the paper.

In [ ]:
ref_feats_c_norm.shape

In [ ]:
src_feats_c_norm.shape

In [ ]:
# 6. Select topk nearest node correspondences
with torch.no_grad():
    ref_node_corr_indices, src_node_corr_indices, node_corr_scores = model.coarse_matching(
        ref_feats_c_norm, src_feats_c_norm, ref_node_masks, src_node_masks
    )

print(f"Number of superpoint correspondences: {ref_node_corr_indices.shape[0]}")

### Visualization: Superpoint Correspondences

Here we visualize the superpoints (red: reference, green: source) and the lines connecting the matched superpoints (blue). This shows the global alignment found by the Geometric Transformer.

In [ ]:
# Visualize Superpoints and Correspondences
ref_pcd_c = make_open3d_point_cloud(ref_points_c.cpu().numpy())
src_pcd_c = make_open3d_point_cloud(src_points_c.cpu().numpy())
ref_pcd_c.paint_uniform_color(get_color('red'))
src_pcd_c.paint_uniform_color(get_color('green'))

# Create lines for correspondences
points_vis = np.concatenate([ref_points_c.cpu().numpy(), src_points_c.cpu().numpy()], axis=0)
lines = []
for i in range(ref_node_corr_indices.shape[0]):
    ref_idx = ref_node_corr_indices[i].item()
    src_idx = src_node_corr_indices[i].item() + ref_points_c.shape[0]
    lines.append([ref_idx, src_idx])

line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(points_vis),
    lines=o3d.utility.Vector2iVector(lines),
)
line_set.paint_uniform_color(get_color('blue'))

print("Red: Reference Superpoints, Green: Source Superpoints, Blue: Correspondences")
# Use open3d_webrtc_draw if available or standard draw
# open3d_webrtc_draw([ref_pcd_c, src_pcd_c, line_set]) 
o3d.visualization.draw_plotly([ref_pcd_c, src_pcd_c, line_set])

## 5. Fine-Level Registration (Patch Matching)

For each matched superpoint pair, we extract the local patch of fine points. We then use Optimal Transport to find correspondences between points in these patches.

- `ref_node_corr_knn_points`: Points in the reference patches.
- `src_node_corr_knn_points`: Points in the source patches.
- `matching_scores`: Scores from Optimal Transport.

Reference: Section 3.4 of the paper.

In [ ]:
# 7.2 Generate batched node points & feats
ref_node_corr_knn_indices = ref_node_knn_indices[ref_node_corr_indices]
src_node_corr_knn_indices = src_node_knn_indices[src_node_corr_indices]
ref_node_corr_knn_masks = ref_node_knn_masks[ref_node_corr_indices]
src_node_corr_knn_masks = src_node_knn_masks[src_node_corr_indices]
ref_node_corr_knn_points = ref_node_knn_points[ref_node_corr_indices]
src_node_corr_knn_points = src_node_knn_points[src_node_corr_indices]

ref_feats_f = feats_f[:ref_length_f]
src_feats_f = feats_f[ref_length_f:]

ref_padded_feats_f = torch.cat([ref_feats_f, torch.zeros_like(ref_feats_f[:1])], dim=0)
src_padded_feats_f = torch.cat([src_feats_f, torch.zeros_like(src_feats_f[:1])], dim=0)
ref_node_corr_knn_feats = index_select(ref_padded_feats_f, ref_node_corr_knn_indices, dim=0)
src_node_corr_knn_feats = index_select(src_padded_feats_f, src_node_corr_knn_indices, dim=0)

# 8. Optimal transport
matching_scores = torch.einsum('bnd,bmd->bnm', ref_node_corr_knn_feats, src_node_corr_knn_feats)
matching_scores = matching_scores / feats_f.shape[1] ** 0.5
matching_scores = model.optimal_transport(matching_scores, ref_node_corr_knn_masks, src_node_corr_knn_masks)

print("Optimal transport matching scores computed.")

### Visualization: Patches

Let's visualize the patches for the first few superpoint correspondences. This shows the local areas being refined.

In [ ]:
# Visualize Patches for the first correspondence
idx = 0
ref_patch = ref_node_corr_knn_points[idx].cpu().numpy()
src_patch = src_node_corr_knn_points[idx].cpu().numpy()
ref_mask = ref_node_corr_knn_masks[idx].cpu().numpy()
src_mask = src_node_corr_knn_masks[idx].cpu().numpy()

# Filter out padded points
ref_patch = ref_patch[:int(ref_mask.sum())]
src_patch = src_patch[:int(src_mask.sum())]

ref_patch_pcd = make_open3d_point_cloud(ref_patch)
src_patch_pcd = make_open3d_point_cloud(src_patch)
ref_patch_pcd.paint_uniform_color(get_color('red'))
src_patch_pcd.paint_uniform_color(get_color('green'))

print(f"Visualizing patch pair {idx}")
o3d.visualization.draw_plotly([ref_patch_pcd, src_patch_pcd])

## 6. Final Transformation Estimation

Finally, we use the fine-level correspondences and their scores to estimate the rigid transformation using a weighted SVD (or RANSAC-like approach if configured).

- `estimated_transform`: The final 4x4 transformation matrix.

In [ ]:
# 9. Generate final correspondences
with torch.no_grad():
    if not model.fine_matching.use_dustbin:
        matching_scores = matching_scores[:, :-1, :-1]

    ref_corr_points, src_corr_points, corr_scores, estimated_transform = model.fine_matching(
        ref_node_corr_knn_points,
        src_node_corr_knn_points,
        ref_node_corr_knn_masks,
        src_node_corr_knn_masks,
        matching_scores,
        node_corr_scores,
    )
print(f"Estimated Transform:\n{estimated_transform.cpu().numpy()}")

### Visualization: Final Registration

Visualizing the source point cloud transformed by the estimated transformation, aligned with the reference point cloud.

In [ ]:
# Visualize Final Alignment
src_pcd = make_open3d_point_cloud(src_points.cpu().numpy())
ref_pcd = make_open3d_point_cloud(ref_points.cpu().numpy())
src_pcd.transform(estimated_transform.cpu().numpy())

ref_pcd.paint_uniform_color(get_color('red'))
src_pcd.paint_uniform_color(get_color('green'))

print("Red: Reference, Green: Transformed Source")
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

---
## DIAGNOSTIC: Ghost zero-points & fixed landmark index validity

Two questions:
1. Does `generate_reference_geometry` produce ghost `(0,0,0)` points for uncovered mesh vertices?
2. Do the 32 `fixed_indices` land on anatomically sensible locations in morphed_ref?

In [3]:
import torch
import numpy as np
import open3d as o3d

# --- Q1: Coverage check ---
pca_data = torch.load("pca_basis_all.pth", weights_only=False)
pca_basis   = pca_data['basis']        # [32, 100, K, 3] or similar
pca_mean    = pca_data['mean']         # [32, K, 3] or similar
patch_indices = pca_data['patch_indices']  # [32, K] — vertex indices into mesh

flat_indices = patch_indices.view(-1).long()
num_global_verts = flat_indices.max().item() + 1
unique_covered = flat_indices.unique()
n_covered = unique_covered.numel()

print(f"patch_indices shape:  {patch_indices.shape}")
print(f"flat_indices max:     {flat_indices.max().item()} → num_global_verts = {num_global_verts}")
print(f"Unique vertices in patches: {n_covered}")
print(f"Ghost (zero-pos) vertices: {num_global_verts - n_covered}  "
      f"({100*(num_global_verts-n_covered)/num_global_verts:.1f}% of morphed_ref will be (0,0,0))")

# --- avg_ref point count from dataset ---
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
from geotransformer.datasets.registration.threedmatch.dataset import ThreeDMatchPairDataset

sample_dataset = ThreeDMatchPairDataset(
    dataset_root=f"../../data/faces",
    subset='val',
    use_augmentation=False,
)
sample = sample_dataset[0]
avg_ref_pts = sample['ref_points']
print(f"\navg_ref from dataset:   {avg_ref_pts.shape[0]} points")
print(f"morphed_ref (generate): {num_global_verts} points")
print(f"  → size mismatch: {avg_ref_pts.shape[0] != num_global_verts}")

patch_indices shape:  torch.Size([32, 700])
flat_indices max:     10787 → num_global_verts = 10788
Unique vertices in patches: 10645
Ghost (zero-pos) vertices: 143  (1.3% of morphed_ref will be (0,0,0))

avg_ref from dataset:   10788 points
morphed_ref (generate): 10788 points
  → size mismatch: False


In [4]:
import torch
import torch.nn.functional as F

# --- Standalone reimplementation of generate_reference_geometry for inspection ---
def generate_morphed_ref(z_delta, pca_basis, pca_mean, patch_indices):
    """Returns (ref_points [N,3], has_points [N bool])"""
    num_patches = patch_indices.shape[0]
    k_neighbors = patch_indices.shape[1]

    delta = torch.matmul(z_delta.unsqueeze(1), pca_basis).squeeze(1)
    reconstructed_patches_flat = pca_mean + delta
    reconstructed_points = reconstructed_patches_flat.view(num_patches, k_neighbors, 3)

    flat_indices = patch_indices.view(-1).long()
    flat_points  = reconstructed_points.contiguous().view(-1, 3)
    num_global_verts = flat_indices.max().item() + 1

    patch_scores = torch.arange(num_patches, dtype=torch.long)
    flat_scores  = patch_scores.unsqueeze(1).expand(-1, k_neighbors).reshape(-1)

    max_scores = torch.full((num_global_verts,), -1, dtype=torch.long)
    max_scores.scatter_reduce_(0, flat_indices, flat_scores, reduce='amax', include_self=False)
    is_max = flat_scores == max_scores[flat_indices]

    valid_positions      = torch.arange(flat_indices.size(0))[is_max]
    valid_global_indices = flat_indices[is_max]

    best_idx = torch.zeros(num_global_verts, dtype=torch.long)
    best_idx.scatter_(0, valid_global_indices, valid_positions)

    ref_points = torch.zeros((num_global_verts, 3))
    has_points = torch.zeros(num_global_verts, dtype=torch.bool)
    has_points[valid_global_indices] = True
    ref_points[has_points] = flat_points[best_idx[has_points]]

    return ref_points, has_points

# Load a val sample and get gt_z
gt_z_raw = torch.load(
    os.path.join(f"../../data/{FACES_FOLDER}", sample['gt_z_path'] if 'gt_z_path' in sample else ""),
    weights_only=False
) if 'gt_z_path' in sample else sample['gt_z']

gt_z = gt_z_raw.float().cpu()
pca_basis_cpu   = pca_data['basis'].float().cpu()    # [32, 100, K*3] or [32, 100, K, 3] — reshape as needed
pca_mean_cpu    = pca_data['mean'].float().cpu()
patch_idx_cpu   = pca_data['patch_indices'].long().cpu()

# Reshape basis/mean to [32, 100, K*3] for matmul if needed
num_p, num_c = gt_z.shape
K3 = pca_basis_cpu.shape[-1] if pca_basis_cpu.ndim == 3 else pca_basis_cpu.shape[2] * pca_basis_cpu.shape[3] * (1 if pca_basis_cpu.ndim == 3 else 3)
if pca_basis_cpu.ndim == 4:
    K = pca_basis_cpu.shape[2]
    pca_basis_cpu = pca_basis_cpu.reshape(num_p, num_c, -1)  # [32, 100, K*3]
    pca_mean_cpu  = pca_mean_cpu.reshape(num_p, -1)           # [32, K*3]

morphed_ref, has_pts = generate_morphed_ref(gt_z, pca_basis_cpu, pca_mean_cpu, patch_idx_cpu)

print(f"morphed_ref shape: {morphed_ref.shape}")
print(f"covered: {has_pts.sum().item()} / {has_pts.shape[0]} ({100*has_pts.float().mean().item():.1f}%)")
print(f"ghost zeros: {(~has_pts).sum().item()}")
if (~has_pts).any():
    print("  ⚠ Ghost zero-points exist — these will contaminate Stage-0 of KPConv graph")

morphed_ref shape: torch.Size([10788, 3])
covered: 10645 / 10788 (98.7%)
ghost zeros: 143
  ⚠ Ghost zero-points exist — these will contaminate Stage-0 of KPConv graph


In [5]:
# --- Q2: Visualize 32 fixed landmark indices on avg_ref vs morphed_ref ---
import open3d as o3d
import numpy as np

FIXED_INDICES = [75, 411, 2699, 911, 8594, 3380, 6731, 9710, 9633, 119,
                 3441, 6319, 9541, 8732, 6162, 3774, 8296, 3151, 10,
                 7720, 6858, 7409, 7531, 3504, 6937, 4189, 8891, 3721,
                 9241, 2213, 1765, 7547]

avg_ref_np = avg_ref_pts  # numpy [N, 3] from dataset
morphed_ref_np = morphed_ref.numpy()  # [num_global_verts, 3]

print(f"avg_ref: {avg_ref_np.shape[0]} pts | morphed_ref: {morphed_ref_np.shape[0]} pts")
print(f"max fixed index: {max(FIXED_INDICES)} | avg_ref size: {avg_ref_np.shape[0]} | valid for avg_ref: {max(FIXED_INDICES) < avg_ref_np.shape[0]}")
print(f"max fixed index: {max(FIXED_INDICES)} | morphed_ref size: {morphed_ref_np.shape[0]} | valid for morphed_ref: {max(FIXED_INDICES) < morphed_ref_np.shape[0]}")

# Check if fixed_index landmarks in morphed_ref are covered (non-zero)
fixed_covered = [has_pts[i].item() for i in FIXED_INDICES]
print(f"\nFixed indices covered by patches: {sum(fixed_covered)}/32")
if not all(fixed_covered):
    bad = [FIXED_INDICES[i] for i, c in enumerate(fixed_covered) if not c]
    print(f"  ⚠ Uncovered (will be zero-pos): {bad}")

# Visualize: avg_ref (gray) + 32 landmarks (red spheres)
avg_pcd = o3d.geometry.PointCloud()
avg_pcd.points = o3d.utility.Vector3dVector(avg_ref_np)
avg_pcd.paint_uniform_color([0.7, 0.7, 0.7])

lm_on_avg = avg_ref_np[FIXED_INDICES]
avg_lm_pcd = o3d.geometry.PointCloud()
avg_lm_pcd.points = o3d.utility.Vector3dVector(lm_on_avg)
avg_lm_pcd.paint_uniform_color([1.0, 0.0, 0.0])

# Visualize: morphed_ref covered pts (blue) + ghost zeros (yellow) + 32 landmarks (red)
covered_pts = morphed_ref_np[has_pts.numpy()]
ghost_pts   = morphed_ref_np[~has_pts.numpy()]

morph_pcd = o3d.geometry.PointCloud()
morph_pcd.points = o3d.utility.Vector3dVector(covered_pts)
morph_pcd.paint_uniform_color([0.2, 0.4, 1.0])

ghost_pcd = o3d.geometry.PointCloud()
ghost_pcd.points = o3d.utility.Vector3dVector(ghost_pts)
ghost_pcd.paint_uniform_color([1.0, 1.0, 0.0])

lm_on_morph = morphed_ref_np[FIXED_INDICES]
morph_lm_pcd = o3d.geometry.PointCloud()
morph_lm_pcd.points = o3d.utility.Vector3dVector(lm_on_morph)
morph_lm_pcd.paint_uniform_color([1.0, 0.0, 0.0])

print("\n--- avg_ref (gray) + 32 landmarks (red) ---")
o3d.visualization.draw_plotly([avg_pcd, avg_lm_pcd])
print("\n--- morphed_ref covered (blue) + ghost zeros (yellow) + 32 landmarks (red) ---")
o3d.visualization.draw_plotly([morph_pcd, ghost_pcd, morph_lm_pcd])

avg_ref: 10788 pts | morphed_ref: 10788 pts
max fixed index: 9710 | avg_ref size: 10788 | valid for avg_ref: True
max fixed index: 9710 | morphed_ref size: 10788 | valid for morphed_ref: True

Fixed indices covered by patches: 32/32

--- avg_ref (gray) + 32 landmarks (red) ---



--- morphed_ref covered (blue) + ghost zeros (yellow) + 32 landmarks (red) ---
